# PPE Detection — End-to-End Test Notebook

**Purpose:** Run tests that require GPU, model weights, and the full
dependency stack (ultralytics, torch, opencv). These cannot run in
GitHub Actions CI because the runners lack GPU hardware.

**How to use:**
1. Open this notebook in [Google Colab](https://colab.research.google.com/).
2. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU).
3. Run all cells. Every cell prints PASS/FAIL at the end.

In [ ]:
# Cell 1: Clone the repo and install dependencies
import os

REPO_URL = "https://github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git"
REPO_DIR = "/content/Worker-Safety-PPE-Detection-Model"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e '.[dev]'
!pip install -q fastapi uvicorn python-multipart httpx
print("\n=== Setup complete ===")

In [ ]:
# Cell 2: Verify GPU and torch availability
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Set runtime to GPU for full testing.")
print("\nPASS: torch imports OK")

In [ ]:
# Cell 3: Run the full CI-compatible test suite (schema + compliance + inference mocks)
!python -m pytest tests/test_schema.py tests/test_compliance.py tests/test_inference.py -v

In [ ]:
# Cell 4: End-to-end model loading and single-image inference
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from ppe.inference import PPEDetector
from ppe.compliance import associate_ppe_to_persons

WEIGHTS = Path("baselines/snehilsanyal_yolov8n_css/models/best.pt")
assert WEIGHTS.is_file(), f"Weights not found: {WEIGHTS}"

device = "cuda" if torch.cuda.is_available() else "cpu"
detector = PPEDetector(str(WEIGHTS), conf=0.25, device=device)
print(f"Model loaded on {device}")
print(f"Model class names: {detector.names()}")

# Predict on a bundled test image
test_image = "baselines/snehilsanyal_yolov8n_css/source_files/construction-safety.jpg"
assert Path(test_image).is_file(), f"Test image not found: {test_image}"

detections = detector.predict_image(test_image)
print(f"\nDetections: {len(detections)}")
for det in detections:
    print(f"  {det.cls_name:15s}  conf={det.conf:.2f}  box={det.xyxy}")

workers = associate_ppe_to_persons(detections)
print(f"\nWorkers: {len(workers)}")
for w in workers:
    print(f"  {w.label}")
    print(f"    present={w.present}  missing={w.missing}  violations={w.violations}")

assert len(detections) > 0, "FAIL: No detections on known image"
print("\nPASS: Single-image inference works")

In [ ]:
# Cell 5: predict_and_comply (annotated image + compliance)
import cv2
import numpy as np

annotated, workers = detector.predict_and_comply(test_image)
assert isinstance(annotated, np.ndarray), "FAIL: annotated should be numpy array"
assert annotated.shape[2] == 3, "FAIL: annotated should be BGR 3-channel"
assert len(workers) >= 0, "FAIL: workers should be a list"

print(f"Annotated image shape: {annotated.shape}")
print(f"Workers found: {len(workers)}")
for w in workers:
    print(f"  {w.label}")

# Display in notebook
from IPython.display import display
from PIL import Image
rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
display(Image.fromarray(rgb).resize((640, int(640 * rgb.shape[0] / rgb.shape[1]))))

print("\nPASS: predict_and_comply produces annotated image")

In [ ]:
# Cell 6: Test all bundled source images
from pathlib import Path

source_dir = Path("baselines/snehilsanyal_yolov8n_css/source_files")
# YOLO-supported formats (jfif not supported; use jpg/jpeg/png/webp only)
image_exts = {".jpg", ".jpeg", ".png", ".webp"}

images = [f for f in source_dir.iterdir() if f.suffix.lower() in image_exts and f.is_file()]
print(f"Found {len(images)} test images")

for img_path in sorted(images):
    dets = detector.predict_image(str(img_path))
    workers = associate_ppe_to_persons(dets)
    status = "OK" if dets else "no-detections"
    print(f"  {img_path.name:50s} → {len(dets):3d} dets, {len(workers):2d} workers  [{status}]")

print("\nPASS: All bundled images processed without errors")

In [ ]:
# Cell 7: FastAPI app smoke test (health + image predict endpoints)
import httpx
from fastapi.testclient import TestClient

# Ensure weights path is set
os.environ["PPE_WEIGHTS"] = str(WEIGHTS.resolve())

from app.api.main import app
client = TestClient(app)

# GET /
r = client.get("/")
assert r.status_code == 200, f"FAIL: GET / returned {r.status_code}"
assert r.json()["service"] == "ppe-detection"
print(f"GET /          → {r.status_code}: {r.json()}")

# GET /health
r = client.get("/health")
assert r.status_code == 200, f"FAIL: GET /health returned {r.status_code}"
body = r.json()
print(f"GET /health    → {r.status_code}: ready={body['ready']}, weights_exists={body['weights_exists']}")
assert body["ready"] is True, f"FAIL: health not ready: {body.get('detail')}"

# POST /predict/image
with open(test_image, "rb") as f:
    r = client.post("/predict/image?conf=0.3", files={"file": ("test.jpg", f, "image/jpeg")})
assert r.status_code == 200, f"FAIL: POST /predict/image returned {r.status_code}: {r.text}"
body = r.json()
print(f"POST /predict  → {r.status_code}: {len(body['detections'])} dets, {len(body['compliance'])} workers")
for label in body["labels"]:
    print(f"  {label}")

print("\nPASS: FastAPI endpoints work")

In [ ]:
# Cell 8: Summary
print("="*60)
print("ALL END-TO-END TESTS PASSED")
print("="*60)
print()
print("Tests covered:")
print("  ✓ CI test suite (schema + compliance + inference mocks)")
print("  ✓ Real model loading on GPU")
print("  ✓ Single-image prediction + compliance")
print("  ✓ predict_and_comply annotated output")
print("  ✓ All bundled source images")
print("  ✓ FastAPI health + predict endpoints")